In [1]:
%matplotlib inline
from pathlib import Path
import sys

PROJECT_ROOT = Path("/data/dn/FRTP_revision1")
IMAGECLS_ROOT = PROJECT_ROOT / "imagecls"
if str(IMAGECLS_ROOT) not in sys.path:
    sys.path.insert(0, str(IMAGECLS_ROOT))

import torch
import torch.nn as nn
import numpy as np

import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from custom_datasets import _make_loader, make_label_drop_subset, make_recovered_target_replay_subset
from FRPT import get_score, save_recons_fea_to_h5

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
DATA_ROOT = PROJECT_ROOT / "mydata"
CKPT_ROOT = IMAGECLS_ROOT / "ckpts" / "cifar100"
RECONS_ROOT = IMAGECLS_ROOT / "recons_data"
print("DEVICE:", DEVICE)


DEVICE: cuda:0


In [2]:
"""数据"""
CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR100_STD = (0.2675, 0.2565, 0.2761)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR100_MEAN, std=CIFAR100_STD),
    transforms.RandomErasing(p=0.25),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR100_MEAN, std=CIFAR100_STD),
])

DATASET_NAME = "cifar100"
trainset = datasets.CIFAR100(root=str(DATA_ROOT), train=True, download=True, transform=transform_train)
testset = datasets.CIFAR100(root=str(DATA_ROOT), train=False, download=True, transform=transform_test)

BATCH_SIZE = 512
NUM_WORKERS = 4

train_loader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True) 
test_loader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False) 



Files already downloaded and verified
Files already downloaded and verified


In [ ]:
def train_base(model, trainloader, testloader, optimizer, num_epochs, es_patience=10, scheduler=None):
    # if scheduler is None:
    #     scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    #         optimizer, mode='max', factor=0.5, patience=3, threshold=0.0, min_lr=1e-6
    #     )
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    test_acc_ls, loss_ls, train_acc_ls = [], [], []
    best_acc, best_ckpt, wait = -1.0, None, 0

    for epoch in range(num_epochs):
        model.train()
        epoch_loss, sample_num = 0.0, 0

        for x, y in trainloader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            loss = criterion(model(x)['out'], y)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            bs = x.size(0)
            epoch_loss += loss.item() * bs
            sample_num += bs

        epoch_loss /= sample_num
        loss_ls.append(epoch_loss)

        test_acc = get_score(model, testloader, DEVICE)
        test_acc_ls.append(test_acc)
        lr = optimizer.param_groups[0]['lr']

        # train_acc_ls.append(eval_acc(model, trainloader)) # use?

        if len(train_acc_ls):
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.6f}, Train Accuracy: {train_acc_ls[-1]:.4f}, Test Accuracy: {test_acc:.4f}, LR: {lr:.2e}')
        else:
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.6f}, Test Accuracy: {test_acc:.4f}, LR: {lr:.2e}')
        
        if ((epoch + 1) % 20 == 0 and (epoch + 1) not in [20,]) or (epoch + 1) == num_epochs:
            periodic_ckpt = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            name = getattr(model, 'name', model.__class__.__name__)
            fname = CKPT_ROOT / f"{name}_e{epoch+1}_{test_acc:.4f}.pth"
            try:
                torch.save(periodic_ckpt, str(fname))
                print(f"Saved periodic checkpoint to {fname}")
            except Exception as e:
                print(f"Failed saving periodic checkpoint: {e}")

        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(test_acc)
            else:
                scheduler.step()
            if test_acc > best_acc:
                best_acc = test_acc
                best_ckpt = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                wait = 0
            else:
                wait += 1
                # if es_patience is not None and wait >= es_patience:
                #     print(f'Early stopping at epoch {epoch+1}. Best Test Accuracy: {best_acc:.4f}')
                #     break
    if scheduler is not None:
        model.load_state_dict(best_ckpt)
    return loss_ls, train_acc_ls, test_acc_ls, best_ckpt


: 

## baseline


In [ ]:
"""train baseline model: CIFAR-style ResNet + SGD cosine recipe"""
from models import SimpleCNN_ci100, ResNet_ci100, SimpleViTv2_ci100
# model = SimpleCNN_ci100(activate=torch.relu, version="v2").to(DEVICE)
# model.load_state_dict(torch.load("/data/dn/FRTP_revision1/imagecls/ckpts/cifar100/ci100_simplecnnv2_relu_epoch10_0.4412.pth", map_location=DEVICE))
# optimizer_ = torch.optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-4)
# EPOCHES = 20
# loss_lt, trainacc_lt, testacc_lt, best_ckpt = train_base(model, train_loader, test_loader, optimizer_, EPOCHES, scheduler=None)

# model = ResNet_ci100(version='18', pretrain=False).to(DEVICE)
# EPOCHES = 200
# optimizer_ = torch.optim.SGD(model.parameters(),lr=0.1,momentum=0.9,weight_decay=5e-4,nesterov=True,)
# scheduler_ = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_, T_max=EPOCHES, eta_min=1e-4)
# loss_lt, trainacc_lt, testacc_lt, best_ckpt = train_base(model,train_loader,test_loader, optimizer_,EPOCHES, scheduler=scheduler_)

model = SimpleViTv2_ci100().to(DEVICE)
model.load_state_dict(torch.load("/data/dn/FRTP_revision1/imagecls/ckpts/cifar100/ci100_simplevitv2_p4_d192_l9_e80_0.4880.pth", map_location=DEVICE))
decay, no_decay = [], []
for name, param in model.named_parameters():
    if not param.requires_grad: continue
    if param.ndim == 1 or name.endswith(".bias"): no_decay.append(param)
    else: decay.append(param)
optimizer_ = torch.optim.AdamW(
    [{"params": decay, "weight_decay": 5e-2}, {"params": no_decay, "weight_decay": 0.0}, ],
    lr=1e-4,  betas=(0.9, 0.999),  eps=1e-8,)
EPOCHS = 200
scheduler_ = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_,T_max=EPOCHS,eta_min=1e-6)
loss_lt, trainacc_lt, testacc_lt, best_ckpt = train_base(model,train_loader,test_loader, optimizer_,EPOCHS, scheduler=scheduler_)

print("baseline_loss.extend(")
print(loss_lt)
print("baseline_testacc.extend(")
print(testacc_lt)

/home/dn/.tmp/ipykernel_511554/1715141348.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("/data/dn/FRTP_revision1/imagecls/ckpts/cifar

Epoch [1/200], Loss: 1.773103, Test Accuracy: 0.4895, LR: 1.00e-04
Epoch [2/200], Loss: 1.754819, Test Accuracy: 0.4938, LR: 1.00e-04
Epoch [3/200], Loss: 1.754876, Test Accuracy: 0.4924, LR: 1.00e-04
Epoch [4/200], Loss: 1.755116, Test Accuracy: 0.4917, LR: 9.99e-05
Epoch [5/200], Loss: 1.755426, Test Accuracy: 0.4899, LR: 9.99e-05
Epoch [6/200], Loss: 1.748466, Test Accuracy: 0.4910, LR: 9.98e-05
Epoch [7/200], Loss: 1.747485, Test Accuracy: 0.4879, LR: 9.98e-05
Epoch [8/200], Loss: 1.736871, Test Accuracy: 0.4933, LR: 9.97e-05
Epoch [9/200], Loss: 1.728132, Test Accuracy: 0.4903, LR: 9.96e-05
Epoch [10/200], Loss: 1.727718, Test Accuracy: 0.4890, LR: 9.95e-05
Epoch [11/200], Loss: 1.736160, Test Accuracy: 0.4923, LR: 9.94e-05
Epoch [12/200], Loss: 1.720955, Test Accuracy: 0.4924, LR: 9.93e-05
Epoch [13/200], Loss: 1.725831, Test Accuracy: 0.4908, LR: 9.91e-05
Epoch [14/200], Loss: 1.713091, Test Accuracy: 0.4899, LR: 9.90e-05
Epoch [15/200], Loss: 1.710860, Test Accuracy: 0.4928, LR

## get recons data


In [6]:
from models import SimpleCNN_ci100, ResNet_ci100, SimpleViT_ci100, SimpleViTv2_ci100
# model = SimpleViTv2_ci100().to(DEVICE)
# MODELPATH = "/data/dn/FRTP_revision1/imagecls/ckpts/cifar100/ci100_simplevit_p4_d192_l9_e30_0.6320.pth"
# model = ResNet_ci100(version='18', pretrain=False).to(DEVICE)
# MODELPATH = "/data/dn/FRTP_revision1/imagecls/ckpts/cifar100/ci100_resnet18_epoch200_0.7926.pth"
model = SimpleCNN_ci100(activate=torch.relu, version="v2").to(DEVICE)
# MODELPATH = "/data/dn/FRTP_revision1/imagecls/ckpts/cifar100/ci100_simplecnnv2_relu_epoch10_0.4386.pth"
# MODELPATH = "/data/dn/FRTP_revision1/imagecls/ckpts/cifar100/ci100_simplecnnv2_relu_0.4411.pth"
MODELPATH = "/data/dn/FRTP_revision1/imagecls/ckpts/cifar100/ci100_simplecnnv2_relu_e5_0.4430.pth"
ckpt = torch.load(MODELPATH, map_location=DEVICE, weights_only=True)
model.load_state_dict(ckpt)
model.eval()
# test_score = get_score(model, test_loader, DEVICE)
# TESTACC = test_score* 100
# train_score = get_score(model, train_loader, DEVICE)
# TRAINACC = train_score* 100
# print(f"testacc={TESTACC:.2f}%, trainacc={TRAINACC:.2f}%")

model.check_cond()

conv1.weight torch.Size([18, 3, 5, 5]) --------------------
tensor([[5, 5, 5],
        [5, 5, 5],
        [5, 5, 5],
        [5, 5, 5],
        [5, 5, 5],
        [5, 5, 5],
        [5, 5, 5],
        [5, 5, 5],
        [5, 5, 5],
        [5, 5, 5],
        [5, 5, 5],
        [5, 5, 5],
        [5, 5, 5],
        [5, 5, 5],
        [5, 5, 5],
        [5, 5, 5],
        [5, 5, 5],
        [5, 5, 5]], device='cuda:7')
tensor([[ 121.4262,   13.1063,   19.0823],
        [  27.0546,   65.0625,  198.3411],
        [  38.5923,    7.1308,   30.5462],
        [ 231.8338,   38.2271,   50.8247],
        [   8.6806,   42.8167,   18.1864],
        [   8.6663,   21.2139,   23.8269],
        [  47.6934,   23.7431,   51.6503],
        [  15.4216,  572.2192,   28.9302],
        [ 912.0516,   35.8653,  138.3243],
        [  39.6356,  108.9974,   69.4833],
        [  11.4285,    7.0188,    7.6412],
        [  14.9657,   29.4891,   15.9187],
        [   8.9531,  154.8412,   22.5510],
        [  22.1945,  

In [7]:
FILEPATH = Path("/data/dn/FRTP_revision1/imagecls/recons_data_new")  / DATASET_NAME / f"{Path(MODELPATH).stem}.h5"
save_recons_fea_to_h5(model, train_loader, str(FILEPATH), DEVICE)

saved recons data to /data/dn/FRTP_revision1/imagecls/recons_data_new/cifar100/ci100_simplecnnv2_relu_e5_0.4430.h5


In [ ]:
from models import SimpleCNN_ci10, ResNet_ci10
import os

for filename in os.listdir("/data/dn/FRTP_revision1/imagecls/ckpts/everyci100"):
    if filename == 'cifar10': continue
    # model = SimpleCNN_ci10(activate=torch.relu, version="v2").to(DEVICE)
    model = ResNet_ci10(version='18', pretrain=False).to(DEVICE)
    MODELPATH = "/data/dn/FRTP_revision1/imagecls/ckpts/everyci10/"+filename
    ckpt = torch.load(MODELPATH, map_location=DEVICE, weights_only=True)
    model.load_state_dict(ckpt)
    model.eval()
    FILEPATH = RECONS_ROOT / 'everyci10' / f"{Path(MODELPATH).stem}.h5"
    if os.path.exists(FILEPATH):
        print("[DONE]", FILEPATH)
        continue
    # FILEPATH = RECONS_ROOT / DATASET_NAME / f"{Path(MODELPATH).stem}.h5"
    save_recons_fea_to_h5(model, train_loader, str(FILEPATH), DEVICE)
    print("saved posttrain recons data to", FILEPATH)

## FRPT


In [ ]:
from models import SimpleCNN_ci100
model = SimpleCNN_ci100(activate=torch.relu, version='v2').to(DEVICE)
MODELPATH = "/data/dn/FRTP_revision1/imagecls/ckpts/cifar100/ci100_simplecnnv2_relu_0.4411.pth"
EPOCHS = 10
SEED_ls = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9][:1]
ALPHA_ls = [0, 0.1]

summary_res = model_post_train_all(model, MODELPATH, test_loader, 
                                ALPHA_ls, SEED_ls, EPOCHS, DEVICE, BATCH_SIZE, save_res=False)

## ablation


In [ ]:
from models import SimpleCNN_ci100
model = SimpleCNN_ci100(activate=torch.relu, version="v1").to(DEVICE)
MODELPATH = "/data/dn/FRTP_revision1/imagecls/ckpts/cifar100/ci100_simplecnnv1_relu_0.3980.pth"
EPOCHS = 10
ALPHA_ls = [0, 0.1, 0.3, 0.5, 0.7, 0.9]
SEED_ls = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9][:1]

ablation_summary = ablation_all(
    model, MODELPATH, test_loader, ALPHA_ls, SEED_ls, EPOCHS, DEVICE, BATCH_SIZE, save_res=False)

## visualize

In [ ]:
from pathlib import Path
from matplotlib import colormaps
from matplotlib.colors import BoundaryNorm
from models import SimpleCNN_ci100

DATASET_NAME = globals().get("DATASET_NAME", "cifar100")
PROJECT_ROOT = Path("/data/dn/FRTP_revision1")
CKPT_ROOT = PROJECT_ROOT / "imagecls" / "ckpts"
VIS_MEAN = torch.tensor((0.5071, 0.4867, 0.4408)).view(3, 1, 1)
VIS_STD = torch.tensor((0.2675, 0.2565, 0.2761)).view(3, 1, 1)
VIS_FEATURE_NAME = "z1"
VIS_SAMPLE_IDX = 0
VIS_TARGET_LABEL = None
VIS_MAX_CHANNELS = 6
VIS_ALPHA = 0.70
VIS_DIFF_PERCENTILE = 0.90
VIS_HEATMAP_LEVELS = 7
VIS_CONV_METHOD = "fft_pad"

vis_model = SimpleCNN_ci100(activate=torch.relu, version="v1").to(DEVICE)
if "MODELPATH" not in globals():
    ckpt_candidates = sorted((CKPT_ROOT / DATASET_NAME).glob(f"{vis_model.name}_*.pth"))
    assert ckpt_candidates, f"No checkpoint found for {vis_model.name}; run baseline cell or set MODELPATH manually."
    MODELPATH = ckpt_candidates[-1]
MODELPATH = Path(MODELPATH)
vis_model.load_state_dict(torch.load(MODELPATH, map_location=DEVICE, weights_only=True))
vis_model.eval()

def denormalize_img(x):
    x = x.detach().cpu().squeeze(0).float()
    if x.dim() == 2:
        x = x.unsqueeze(0)
    if x.shape[0] == 3:
        x = x * VIS_STD + VIS_MEAN
    x = x.clamp(0, 1)
    return x.permute(1, 2, 0).numpy() if x.shape[0] == 3 else x.squeeze(0).numpy()

def normalize_map(x):
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().float()
    x_min, x_max = x.min(), x.max()
    return (x - x_min) / (x_max - x_min + 1e-12)

def pick_visual_sample(model, dataset, sample_idx=0, target_label=None, device=DEVICE):
    if target_label is None:
        x, y = dataset[sample_idx]
        x_dev = x.unsqueeze(0).to(device)
        with torch.no_grad():
            logits = model(x_dev)["out"].detach().cpu()
        pred = int(logits.argmax(dim=1).item())
        return sample_idx, x_dev, int(y), pred, logits

    loader = DataLoader(dataset, batch_size=1, shuffle=False)
    model.eval()
    with torch.no_grad():
        for idx, (x, y) in enumerate(loader):
            if int(y.item()) != target_label:
                continue
            x_dev = x.to(device)
            logits = model(x_dev)["out"].detach().cpu()
            pred = int(logits.argmax(dim=1).item())
            return idx, x_dev, int(y.item()), pred, logits
    raise RuntimeError(f"No sample found for target_label={target_label}")

def visualize_forward_recons_diff(model, dataset, feature_name="z1", sample_idx=0, target_label=None):
    idx, input_data, label, pred, logits = pick_visual_sample(model, dataset, sample_idx, target_label)
    with torch.no_grad():
        forward_res = model(input_data)
        recons_res = model.get_recons_fea(input_data, torch.tensor([label], device=input_data.device), conv_method=VIS_CONV_METHOD)

    if feature_name == "out":
        forward_feature = forward_res["out"].detach().cpu().flatten()
        recons_feature = recons_res["recons_out"].detach().cpu().flatten()
        diff = recons_feature - forward_feature
        fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
        axes[0].imshow(denormalize_img(input_data))
        axes[0].set_title(f"idx={idx}, gt={label}, pred={pred}")
        axes[0].axis("off")
        axes[1].plot(forward_feature.numpy(), label="forward")
        axes[1].plot(recons_feature.numpy(), label="recons")
        axes[1].plot(diff.numpy(), label="diff")
        axes[1].legend()
        axes[1].set_title("out / recons_out")
        plt.show()
        return

    forward_feature = forward_res[feature_name].detach().cpu().squeeze(0)
    recons_feature = recons_res[f"recons_{feature_name}"].detach().cpu().squeeze(0)
    assert forward_feature.shape == recons_feature.shape, (forward_feature.shape, recons_feature.shape)
    diff = recons_feature - forward_feature
    diff_quantile = VIS_DIFF_PERCENTILE / 100 if VIS_DIFF_PERCENTILE > 1 else VIS_DIFF_PERCENTILE
    vmax = torch.quantile(diff.abs().flatten(), diff_quantile).item()
    vmax = max(vmax, diff.abs().max().item() * 1e-6, 1e-12)
    channel_count = min(forward_feature.size(0), VIS_MAX_CHANNELS)
    bounds = np.linspace(-vmax, vmax, VIS_HEATMAP_LEVELS + 1)
    cmap = colormaps["bwr"].resampled(VIS_HEATMAP_LEVELS)
    norm = BoundaryNorm(bounds, cmap.N, clip=True)

    fig, axes = plt.subplots(3, channel_count + 1, figsize=(3.0 * (channel_count + 1), 8.5), squeeze=False)
    img = denormalize_img(input_data)
    for r, title in enumerate(["input / forward", "recons", "forward + diff"]):
        axes[r, 0].imshow(img)
        axes[r, 0].set_title(title if r else f"idx={idx}\ngt={label}, pred={pred}")
        axes[r, 0].axis("off")

    im = None
    for c in range(channel_count):
        f_base = normalize_map(forward_feature[c]).numpy()
        r_base = normalize_map(recons_feature[c]).numpy()
        d = diff[c].numpy()
        axes[0, c + 1].imshow(f_base, cmap="gray")
        axes[0, c + 1].set_title(f"forward {feature_name} ch{c}")
        axes[1, c + 1].imshow(r_base, cmap="gray")
        axes[1, c + 1].set_title(f"recons {feature_name} ch{c}")
        axes[2, c + 1].imshow(f_base, cmap="gray")
        im = axes[2, c + 1].imshow(d, cmap=cmap, norm=norm, alpha=VIS_ALPHA)
        axes[2, c + 1].set_title(f"diff ch{c}")
        for r in range(3):
            axes[r, c + 1].axis("off")

    fig.colorbar(im, ax=axes[:, 1:].ravel().tolist(), fraction=0.025, pad=0.02).set_label("recons - forward")
    fig.suptitle(f"{DATASET_NAME}: forward / reconstruction feature comparison ({MODELPATH.name})")
    plt.show()
    print(f"sample_idx={idx}, gt={label}, pred={pred}, logits_head={logits.numpy().round(3).tolist()[0][:10]}")
    print(f"diff mean={diff.mean().item():.6f}, min={diff.min().item():.6f}, max={diff.max().item():.6f}")

visualize_forward_recons_diff(vis_model, testset, VIS_FEATURE_NAME, VIS_SAMPLE_IDX, VIS_TARGET_LABEL)


In [ ]:
import torch.nn.functional as F

CAM_ALPHA = 0.55
CAM_CMAP = "jet"
CAM_DIFF_FEATURE_NAME = "out"      # 可选: "out" 或 model.get_fea_name() 对应的 z 特征，如 "z1", "z2"
CAM_RECONS_FEATURE_NAME = "out"
CAM_FORWARD_FEATURE_NAME = "out"
CAM_SAMPLE_IDX = VIS_SAMPLE_IDX if "VIS_SAMPLE_IDX" in globals() else 0
CAM_TARGET_LABEL = VIS_TARGET_LABEL if "VIS_TARGET_LABEL" in globals() else None

def tensor_to_numpy_img(x):
    return denormalize_img(x)

def upsample_cam(cam, input_data):
    cam = F.interpolate(cam, size=input_data.shape[-2:], mode="bilinear", align_corners=False)
    cam = cam.squeeze().detach().cpu().float()
    return normalize_map(cam).numpy()

def get_diff_driven_cam(model, input_data, label, feature_name="z1"):
    model.eval()
    recons_key = f"recons_{feature_name}"
    if feature_name == "out":
        x = input_data.detach().clone().requires_grad_(True)
        model.zero_grad(set_to_none=True)
        forward_out = model(x)["out"]
        recons_out = model.get_recons_fea(x, torch.tensor([label], device=x.device), conv_method=VIS_CONV_METHOD)["recons_out"]
        diff = recons_out - forward_out
        score = diff[0, label]
        score.backward()
        saliency = x.grad.detach().abs().amax(dim=1, keepdim=True)
        return upsample_cam(saliency, input_data), diff.detach().cpu()

    with torch.no_grad():
        forward_res = model(input_data)
        recons_res = model.get_recons_fea(input_data, torch.tensor([label], device=input_data.device), conv_method=VIS_CONV_METHOD)
    forward_feature = forward_res[feature_name]
    recons_feature = recons_res[recons_key]
    diff = recons_feature - forward_feature
    if diff.dim() != 4:
        raise ValueError(f"diff for {feature_name} has shape {tuple(diff.shape)}; expected 4D or feature_name='out'.")
    weights = diff.mean(dim=(2, 3), keepdim=True)
    cam_signed = (weights * forward_feature).sum(dim=1, keepdim=True)
    cam = F.relu(cam_signed)
    if cam.max().item() <= 1e-12:
        cam = cam_signed.abs()
    return upsample_cam(cam, input_data), diff.detach().cpu()

def feature_map_to_cam(feature, input_data):
    weights = feature.mean(dim=(2, 3), keepdim=True)
    cam_signed = (weights * feature).sum(dim=1, keepdim=True)
    cam = F.relu(cam_signed)
    if cam.max().item() <= 1e-12:
        cam = cam_signed.abs()
    return upsample_cam(cam, input_data)

def get_recons_feature_cam(model, input_data, label, feature_name="z1"):
    if feature_name == "out":
        x = input_data.detach().clone().requires_grad_(True)
        model.zero_grad(set_to_none=True)
        recons_out = model.get_recons_fea(x, torch.tensor([label], device=x.device), conv_method=VIS_CONV_METHOD)["recons_out"]
        score = recons_out[0, label]
        score.backward()
        saliency = x.grad.detach().abs().amax(dim=1, keepdim=True)
        return upsample_cam(saliency, input_data)
    with torch.no_grad():
        recons_feature = model.get_recons_fea(input_data, torch.tensor([label], device=input_data.device), conv_method=VIS_CONV_METHOD)[f"recons_{feature_name}"]
    if recons_feature.dim() != 4:
        raise ValueError(f"recons_{feature_name} has shape {tuple(recons_feature.shape)}; expected 4D or feature_name='out'.")
    return feature_map_to_cam(recons_feature, input_data)

def get_forward_feature_cam(model, input_data, feature_name="z1", class_idx=None):
    if feature_name == "out":
        if class_idx is None:
            with torch.no_grad():
                class_idx = int(model(input_data)["out"].argmax(dim=1).item())
        x = input_data.detach().clone().requires_grad_(True)
        model.zero_grad(set_to_none=True)
        score = model(x)["out"][0, class_idx]
        score.backward()
        saliency = x.grad.detach().abs().amax(dim=1, keepdim=True)
        return upsample_cam(saliency, input_data)
    with torch.no_grad():
        forward_feature = model(input_data)[feature_name]
    if forward_feature.dim() != 4:
        raise ValueError(f"forward feature {feature_name} has shape {tuple(forward_feature.shape)}; expected 4D or feature_name='out'.")
    return feature_map_to_cam(forward_feature, input_data)

def visualize_input_cam_compare(model, dataset, sample_idx=0, target_label=None, diff_feature_name="out", recons_feature_name="out", forward_feature_name="out"):
    idx, input_data, label, pred, logits = pick_visual_sample(model, dataset, sample_idx, target_label)
    diff_cam, diff = get_diff_driven_cam(model, input_data, label, diff_feature_name)
    recons_cam = get_recons_feature_cam(model, input_data, label, recons_feature_name)
    forward_cam = get_forward_feature_cam(model, input_data, forward_feature_name, class_idx=pred)
    input_img = tensor_to_numpy_img(input_data)

    fig, axes = plt.subplots(1, 4, figsize=(14.0, 3.6), squeeze=False)
    axes[0, 0].imshow(input_img)
    axes[0, 0].set_title(f"input\nidx={idx}, gt={label}, pred={pred}")
    axes[0, 0].axis("off")
    for ax, cam, title in [
        (axes[0, 1], diff_cam, f"diff-driven\nfeature={diff_feature_name}"),
        (axes[0, 2], recons_cam, f"recons-feature\nfeature={recons_feature_name}"),
        (axes[0, 3], forward_cam, f"forward-feature\nfeature={forward_feature_name}"),
    ]:
        ax.imshow(input_img)
        im = ax.imshow(cam, cmap=CAM_CMAP, alpha=CAM_ALPHA, vmin=0, vmax=1)
        ax.set_title(title)
        ax.axis("off")
    fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.025, pad=0.02)
    fig.suptitle(f"{DATASET_NAME} input-space highlight comparison ({MODELPATH.name})")
    plt.show()
    print(f"sample_idx={idx}, gt={label}, pred={pred}, logits_head={logits.numpy().round(3).tolist()[0][:10]}")
    print(f"diff mean={diff.mean().item():.6f}, min={diff.min().item():.6f}, max={diff.max().item():.6f}")

visualize_input_cam_compare(
    vis_model, testset, CAM_SAMPLE_IDX, CAM_TARGET_LABEL,
    CAM_DIFF_FEATURE_NAME, CAM_RECONS_FEATURE_NAME, CAM_FORWARD_FEATURE_NAME
)


## different stage vs recons loss

In [ ]:
from models import SimpleCNN_ci100, ResNet_ci100, SimpleViT_ci100
train_loader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False)
print(len(train_loader), len(test_loader))

STAGE_RECONS_MODEL_BUILDERS = {
    "simplecnnv1": lambda: SimpleCNN_ci100(activate=torch.relu, version="v1"),
    "simplecnnv2": lambda: SimpleCNN_ci100(activate=torch.relu, version="v2"),
    "resnet18": lambda: ResNet_ci100(version="18", pretrain=False),
    "simplevit": lambda: SimpleViT_ci100(),
}

def get_score(model, data_loader, DEVICE=DEVICE):
    model.eval()
    correct = 0.0
    with torch.no_grad():
        for data, target in data_loader:
            data = data.to(DEVICE, non_blocking=True)
            target = target.to(DEVICE, non_blocking=True)
            output = model(data)["out"]
            pred = output.argmax(dim=1)
            correct += pred.eq(target.view_as(pred)).sum().item()
    return correct / len(data_loader.dataset)


def get_recons_loss(model, data_loader, DEVICE=DEVICE):
    """Return the sample-averaged reconstruction loss on data_loader."""
    model.eval()
    sample_num = 0
    recons_loss = {recons_key: 0.0 for recons_key in model.get_fea_name()}
    with torch.no_grad():
        for input, target in data_loader:
            input = input.to(DEVICE, non_blocking=True)
            target = target.to(DEVICE, non_blocking=True)
            recons_feature = model.get_recons_fea(input.detach(), target, recons_key=None)
            res = model(input)
            bs = input.size(0)
            sample_num += bs
            for recons_key in model.get_fea_name():
                diff = res[recons_key[7:]] - recons_feature[recons_key]
                reconsloss_batch = diff.flatten(1).pow(2).sum(dim=1).sum()
                recons_loss[recons_key] += reconsloss_batch.item()
    return {recons_key: loss / sample_num for recons_key, loss in recons_loss.items()}


def train_stage_recons(model, lr, num_epochs, train_record=True, every_epoch=5):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    clscrit = nn.CrossEntropyLoss()
    recons_keys = model.get_fea_name()

    train_score_list, test_score_list = [], []
    train_recons_loss_list = {recons_key: [] for recons_key in recons_keys}
    test_recons_loss_list = {recons_key: [] for recons_key in recons_keys}

    for epoch in range(num_epochs):
        if epoch % every_epoch == 0:
            print("************* ", end="")
            test_score = get_score(model, test_loader, DEVICE)
            test_score_list.append(test_score)
            test_recons_loss = get_recons_loss(model, test_loader, DEVICE)
            for recons_key in recons_keys:
                test_recons_loss_list[recons_key].append(test_recons_loss[recons_key])

            if train_record:
                train_score = get_score(model, train_loader, DEVICE)
                train_score_list.append(train_score)
                train_recons_loss = get_recons_loss(model, train_loader, DEVICE)
                for recons_key in recons_keys:
                    train_recons_loss_list[recons_key].append(train_recons_loss[recons_key])
                print(
                    f"Epoch [{epoch + 1}/{num_epochs}] "
                    f"train_score={train_score:.4f}, test_score={test_score:.4f}, "
                    f"train_recons_loss={train_recons_loss}, test_recons_loss={test_recons_loss}"
                )
            else:
                print(
                    f"Epoch [{epoch + 1}/{num_epochs}] test_score={test_score:.4f}, "
                    f"test_recons_loss={test_recons_loss}"
                )

        model.train()
        for data, target in train_loader:
            data = data.to(DEVICE, non_blocking=True)
            target = target.to(DEVICE, non_blocking=True)
            loss = clscrit(model(data)["out"], target)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    print("************* ", end="")
    test_score = get_score(model, test_loader, DEVICE)
    test_score_list.append(test_score)
    test_recons_loss = get_recons_loss(model, test_loader, DEVICE)
    for recons_key in recons_keys:
        test_recons_loss_list[recons_key].append(test_recons_loss[recons_key])
    if train_record:
        train_score = get_score(model, train_loader, DEVICE)
        train_score_list.append(train_score)
        train_recons_loss = get_recons_loss(model, train_loader, DEVICE)
        for recons_key in recons_keys:
            train_recons_loss_list[recons_key].append(train_recons_loss[recons_key])
        print(
            f"Epoch [{epoch + 1}/{num_epochs}] "
            f"train_score={train_score:.4f}, test_score={test_score:.4f}, "
            f"train_recons_loss={train_recons_loss}, test_recons_loss={test_recons_loss}"
        )
    else:
        print( f"Epoch [{epoch + 1}/{num_epochs}] test_score={test_score:.4f}, "
               f"test_recons_loss={test_recons_loss}")
    results = {
        "model": model.name,
        "lr": lr,
        "num_epochs": num_epochs,
        "recons_keys": recons_keys,
        "train_score_list": train_score_list,
        "test_score_list": test_score_list,
        "train_recons_loss_list": train_recons_loss_list,
        "test_recons_loss_list": test_recons_loss_list,
    }

    ckpt_path = Path(f"/data/dn/FRTP_revision1/imagecls/logs_stage/{model.name}_{lr}lr_{num_epochs}e.pth")
    ckpt_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, ckpt_path)
    print(f"saved model ckpt to {ckpt_path}")

    save_path = Path(f"/data/dn/FRTP_revision1/imagecls/logs_stage/{model.name}_{lr}lr_{num_epochs}e.pt")
    save_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(results, save_path)
    print(f"saved results to {save_path}")
    print("train_score_list =", train_score_list)
    print("test_score_list =", test_score_list)
    print("train_recons_loss_list =", train_recons_loss_list)
    print("test_recons_loss_list =", test_recons_loss_list)
    return results


In [ ]:
MODEL_NAME = "simplecnnv2"
MODELPATH = None
LR = 1e-3
EPOCHS = 60
EVERY_EPOCH = 10
TRAIN_RECORD = True

model = STAGE_RECONS_MODEL_BUILDERS[MODEL_NAME]().to(DEVICE)
if MODELPATH is not None:
    ckpt = torch.load(MODELPATH, map_location=DEVICE, weights_only=True)
    if isinstance(ckpt, dict) and "state_dict" in ckpt:
        ckpt = ckpt["state_dict"]
    model.load_state_dict(ckpt)

stage_recons_results = train_stage_recons(
    model,
    lr=LR,
    num_epochs=EPOCHS,
    train_record=TRAIN_RECORD,
    every_epoch=EVERY_EPOCH,
)
train_score_list = stage_recons_results["train_score_list"]
test_score_list = stage_recons_results["test_score_list"]
train_recons_loss_list = stage_recons_results["train_recons_loss_list"]
test_recons_loss_list = stage_recons_results["test_recons_loss_list"]
